# MCBench: Measuring Metacognitive Capability in AI via Signal Detection Theory

**Hackathon:** Measuring Progress Toward AGI – Cognitive Abilities  
**Author:** Ivo Georgiev  
**Dataset:** MMLU-Pro + GPQA Diamond (600 eval items)

## What MCBench measures

MCBench measures **metacognitive capability** — not just whether a model gets answers right,
but whether it *knows what it knows*. This is a key AGI milestone: human-level metacognition
allows us to allocate cognitive effort, recognise uncertainty, and avoid confident mistakes.

### Two-turn evaluation protocol
| Turn | Task | What it measures |
|------|------|------------------|
| 1 | Zero-shot MCQ answer | Type 1 accuracy → d′ |
| 2 | Confidence elicitation (0–100) | Metacognitive sensitivity → meta-d′ |

### Metacognitive Capability Index (MCI)
$$\text{MCI} = 0.65 \cdot \max(0, M\text{-ratio}) + 0.35 \cdot (1 - \text{ECE})$$

where $M\text{-ratio} = \text{meta-}d' / d'$ (Fleming & Lau 2014 via Type 2 AUROC).

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, re
from pathlib import Path

import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np

# Add dataset root to path so src/ is importable on Kaggle
REPO_ROOT = Path('/kaggle/input/mcbench-metacognitive-benchmark')
if not REPO_ROOT.exists():
    REPO_ROOT = Path('.').resolve()   # local fallback
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / 'data'
eval_df  = pd.read_parquet(DATA_DIR / 'eval_set.parquet')

print(f'Repo root : {REPO_ROOT}')
print(f'Eval set  : {len(eval_df)} items')
print(f'Domains   : {eval_df["domain"].value_counts().to_dict()}')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def _extract_confidence(text: str) -> float:
    """Return the last 0-100 integer found in text, defaulting to 50."""
    for c in reversed(re.findall(r'\b(\d{1,3})\b', text)):
        val = int(c)
        if 0 <= val <= 100:
            return float(val)
    return 50.0


def _judge_accuracy(response: str, correct_answer: str) -> int:
    """Return 1 if the correct answer letter appears in the response."""
    return int(bool(
        re.search(rf'\b{re.escape(correct_answer.strip())}\b',
                  response, re.IGNORECASE)
    ))

In [ ]:
# ── MCBench task ──────────────────────────────────────────────────────────────
@kbench.task(name="mcbench_metacognition")
def mcbench_metacognition(llm, df: pd.DataFrame) -> float:
    """
    Two-turn metacognitive evaluation over the MCBench eval set.

    Turn 1: LLM answers the MCQ (accuracy scored against ground truth)
    Turn 2: LLM states confidence 0-100 after reviewing its own answer

    Returns MCI (Metacognitive Capability Index).
    Higher = stronger metacognitive capability.
    """
    from src.metrics.sdt import compute_m_ratio
    from src.metrics.mci import compute_mci

    accuracy_list   = []
    confidence_list = []

    with kbench.client.enable_cache():
        for _, row in df.iterrows():

            # Turn 1: zero-shot MCQ answer
            response1  = llm.prompt(row['main_prompt'])
            is_correct = _judge_accuracy(response1, row['correct_answer'])

            # Turn 2: metacognitive confidence elicitation
            # Present the original question + the model's own answer, then
            # ask for confidence — focuses the model on self-assessment.
            turn2 = (
                row['main_prompt']
                + f"\n\nYour previous answer was:\n{response1}\n\n"
                + row['meta_question']
            )
            response2  = llm.prompt(turn2)
            confidence = _extract_confidence(response2)

            accuracy_list.append(is_correct)
            confidence_list.append(confidence)

    # SRS (self-recognition) requires multi-model decoys; use random baseline
    # 1/3 so MCI remains comparable across models on this leaderboard.
    mci_result = compute_mci(accuracy_list, confidence_list, srs=1/3)

    # Persist detail for post-run calibration plots
    detail = df[['id', 'domain', 'source']].copy()
    detail['accuracy']   = accuracy_list
    detail['confidence'] = confidence_list
    detail.to_csv('mcbench_detail.csv', index=False)

    print(f"\nMCI     = {mci_result['mci']:.4f}")
    print(f"d'      = {mci_result['d_prime']:.4f}")
    print(f"M-ratio = {mci_result['m_ratio']:.4f}")
    print(f"ECE     = {mci_result['ece']:.4f}")
    print(f"Accuracy = {sum(accuracy_list)/len(accuracy_list):.3f}")

    return float(mci_result['mci'])

In [ ]:
# ── Select this as the primary benchmark task ─────────────────────────────────
%choose mcbench_metacognition

In [ ]:
# ── Run the benchmark ─────────────────────────────────────────────────────────
# kbench.llm is the model selected via the Kaggle model picker.
# For a quick smoke test, replace eval_df with eval_df.head(50)
mci_score = mcbench_metacognition.run(kbench.llm, eval_df)
print(f'\nFinal MCI score: {mci_score:.4f}')

In [ ]:
# ── Results summary ───────────────────────────────────────────────────────────
from src.metrics.sdt import compute_m_ratio

detail = pd.read_csv('mcbench_detail.csv')
acc    = detail['accuracy'].tolist()
conf   = detail['confidence'].tolist()
sdt    = compute_m_ratio(acc, conf)

summary = pd.DataFrame([{
    'model':        str(kbench.llm),
    'mci':          mci_score,
    'd_prime':      sdt['d_prime'],
    'm_ratio':      sdt['m_ratio'],
    'type2_auroc':  sdt['type2_auroc'],
    'accuracy':     sum(acc) / len(acc),
    'mean_conf':    sum(conf) / len(conf),
}])

display(summary.style
    .format({c: '{:.4f}' for c in ['mci','d_prime','m_ratio','type2_auroc','accuracy','mean_conf']})
    .set_caption('MCBench Result')
)

summary.to_csv('mcbench_leaderboard.csv', index=False)
print('Saved: mcbench_leaderboard.csv  mcbench_detail.csv')

In [ ]:
# ── Calibration plots ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('MCBench: Metacognitive Capability Profile', fontsize=12)

# Confidence distributions: correct vs incorrect
correct_conf   = [c for c, a in zip(conf, acc) if a == 1]
incorrect_conf = [c for c, a in zip(conf, acc) if a == 0]
axes[0].hist(correct_conf,   bins=20, alpha=0.6, color='steelblue', label='Correct')
axes[0].hist(incorrect_conf, bins=20, alpha=0.6, color='tomato',    label='Incorrect')
axes[0].set_xlabel('Stated Confidence (0–100)')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence by Answer Correctness')
axes[0].legend()
axes[0].axvline(50, color='grey', linestyle='--', alpha=0.5)

# Reliability diagram (calibration curve)
bins_edges = np.linspace(0, 100, 11)
conf_arr   = np.array(conf)
acc_arr    = np.array(acc, dtype=float)
bin_centers, bin_acc = [], []
for lo, hi in zip(bins_edges[:-1], bins_edges[1:]):
    mask = (conf_arr >= lo) & (conf_arr < hi)
    if mask.sum() > 0:
        bin_centers.append((lo + hi) / 2)
        bin_acc.append(acc_arr[mask].mean() * 100)
axes[1].plot(bin_centers, bin_acc, 'o-', color='darkorange', label='Model')
axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Perfect calibration')
axes[1].set_xlabel('Stated Confidence (0–100)')
axes[1].set_ylabel('Actual Accuracy (%)')
axes[1].set_title(f"Calibration Curve  (M-ratio={sdt['m_ratio']:.3f})")
axes[1].legend()

plt.tight_layout()
plt.savefig('mcbench_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mcbench_calibration.png')